<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/6_Correlaci%C3%B3n_Perason_Spearman_Sistemas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de correlación Perason y Spearman

Este notebook calcula la correlación entre la máxima persistencia de H₁ y el exponente de Lyapunov máximo (λ₁) para los sistemas Hopf normal, Lorenz y BZ. Compara las métricas topológicas con las dinámicas mediante correlación de Pearson y Spearman. Exporta resultados en Excel, CSV y formato LaTeX.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# ============================================================
# 1. CARGA DE DATOS DESDE XLSX
# ============================================================

# -------- Max persistence --------
df_mp_hopf = pd.read_excel(r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\HOPF NORMAL\hopf_normal_max_persistence_mu_(-1.0,1.0)_N=300_Tau=26_m=2_noise=0.0.xlsx")
df_mp_lorenz = pd.read_excel(r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\LORENZ\Lorenz_max_persistence_rho_(20.0,30.0)_N=300_Tau=16_m=2_noise=0.0.xlsx")
df_mp_bz = pd.read_excel(r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\REACCIÓN BZ\Reacción_BZ_max_persistence_b_(2.0,5.0)_N=300_Tau=57_m=2_noise=0.0.xlsx")

# -------- Lyapunov --------
df_lyap_hopf = pd.read_excel(r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\HOPF NORMAL\lyapunov_lambda1_Hopf_Normal_mu_((-1.0, 1.0))_300_omega=6.0.xlsx")
df_lyap_lorenz = pd.read_excel(r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\LORENZ\resultados_lyapunov_Lorenz_20-30_T_300_dt0.01_1.0_1300.xlsx")
df_lyap_bz = pd.read_excel(r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\REACCIÓN BZ\resultados_lyapunov_Reacción_BZ_2-5_10_300_dt0.01_1.0_1300.xlsx")

# Normalizar nombres de columnas
for df in [df_mp_hopf, df_mp_lorenz, df_mp_bz, df_lyap_hopf, df_lyap_lorenz, df_lyap_bz]:
    df.columns = [str(c).strip().lower() for c in df.columns]
# Después de normalizar nombres, imprime TODAS las columnas:
print("Hopf MP columns:", df_mp_hopf.columns.tolist())
print("Hopf Lyap columns:", df_lyap_hopf.columns.tolist())
print("Lorenz MP columns:", df_mp_lorenz.columns.tolist())
print("Lorenz Lyap columns:", df_lyap_lorenz.columns.tolist())
print("BZ MP columns:", df_mp_bz.columns.tolist())
print("BZ Lyap columns:", df_lyap_bz.columns.tolist())
# ============================================================
# 2. RENOMBRAR COLUMNAS A FORMATO ESTANDAR
# ============================================================

# -------- Hopf --------
df_mp_hopf = df_mp_hopf.rename(columns={
    "mu": "parameter",
    "max_persistence_H1": "max_persistence_h1"
})
df_lyap_hopf = df_lyap_hopf.rename(columns={
    "mu": "parameter",
    "lambda_1": "lambda_1"
})

# -------- Lorenz --------
df_mp_lorenz = df_mp_lorenz.rename(columns={
    "rho": "parameter",
    "max_persistence_H1": "max_persistence_h1"   # sin ruido
})
df_lyap_lorenz = df_lyap_lorenz.rename(columns={
    "rho": "parameter",
    "lambda_1": "lambda_1"
})

# -------- BZ --------
df_mp_bz = df_mp_bz.rename(columns={
    "b": "parameter",
    "max_persistence_H1": "max_persistence_h1"   # sin ruido
})
df_lyap_bz = df_lyap_bz.rename(columns={
    "b": "parameter",
    "lambda_1": "lambda_1"
})

# ============================================================
# 3. REDONDEAR PARAMETROS PARA HACER MERGE ROBUSTO
# ============================================================
# Esto evita problemas por diferencias minimas entre grids
for df in [df_mp_hopf, df_mp_lorenz, df_mp_bz, df_lyap_hopf, df_lyap_lorenz, df_lyap_bz]:
    df["parameter"] = pd.to_numeric(df["parameter"], errors="coerce").round(3)

# ============================================================
# 4. MERGE POR PARAMETRO
# ============================================================
df_hopf_corr = pd.merge(df_mp_hopf, df_lyap_hopf, on="parameter", how="outer")
df_lorenz_corr = pd.merge(df_mp_lorenz, df_lyap_lorenz, on="parameter", how="outer")
df_bz_corr = pd.merge(df_mp_bz, df_lyap_bz, on="parameter", how="outer")

# ============================================================
# 5. LIMPIEZA
# ============================================================
def clean_corr_df(df, parameter_name="parameter",
                  functional_name="max_persistence_h1",
                  lyap_name="lambda_1"):
    required = [parameter_name, functional_name, lyap_name]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Falta la columna requerida: '{c}'")

    out = df[required].copy()
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    out = out.sort_values(parameter_name).reset_index(drop=True)
    return out

df_hopf_corr = clean_corr_df(df_hopf_corr)
df_lorenz_corr = clean_corr_df(df_lorenz_corr)
df_bz_corr = clean_corr_df(df_bz_corr)

# ============================================================
# 6. DEBUG RAPIDO
# ============================================================
print("Hopf:", df_hopf_corr.shape)
print(df_hopf_corr.head(), "\n")

print("Lorenz:", df_lorenz_corr.shape)
print(df_lorenz_corr.head(), "\n")

print("BZ:", df_bz_corr.shape)
print(df_bz_corr.head(), "\n")

# ============================================================
# 7. FUNCION PARA CALCULAR CORRELACIONES
# ============================================================
def compute_correlation_row(system_name, df,
                            functional_col="max_persistence_h1",
                            lyap_col="lambda_1"):
    x = df[functional_col].values
    y = df[lyap_col].values

    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)

    return {
        "System": system_name,
        "Pearson r": pearson_r,
        "Pearson p-value": pearson_p,
        "Spearman rho": spearman_rho,
        "Spearman p-value": spearman_p,
        "N": len(df)
    }

# ============================================================
# 8. CONSTRUIR TABLA AUTOMATICAMENTE
# ============================================================
rows = []
rows.append(compute_correlation_row("Hopf normal form", df_hopf_corr))
rows.append(compute_correlation_row("Lorenz", df_lorenz_corr))
rows.append(compute_correlation_row("BZ", df_bz_corr))

corr_table = pd.DataFrame(rows)

# Version redondeada para mostrar
corr_table_display = corr_table.copy()
for col in ["Pearson r", "Pearson p-value", "Spearman rho", "Spearman p-value"]:
    corr_table_display[col] = corr_table_display[col].map(lambda v: f"{v:.6g}")

print("\n=== Correlation table: max persistence vs largest Lyapunov exponent ===")
print(corr_table_display.to_string(index=False))

# ============================================================
# 9. EXPORTAR TABLA
# ============================================================
corr_table.to_excel("correlation_max_persistence_vs_lyapunov.xlsx", index=False)
corr_table.to_csv("correlation_max_persistence_vs_lyapunov.csv", index=False)

# ============================================================
# 10. GENERAR TABLA LaTeX
# ============================================================
latex_table = corr_table.copy()
latex_table["Pearson r"] = latex_table["Pearson r"].map(lambda v: f"{v:.4f}")
latex_table["Pearson p-value"] = latex_table["Pearson p-value"].map(lambda v: f"{v:.2e}")
latex_table["Spearman rho"] = latex_table["Spearman rho"].map(lambda v: f"{v:.4f}")
latex_table["Spearman p-value"] = latex_table["Spearman p-value"].map(lambda v: f"{v:.2e}")

latex_code = latex_table.to_latex(index=False, escape=False)

print("\n=== LaTeX table ===")
print(latex_code)

with open("correlation_max_persistence_vs_lyapunov.tex", "w", encoding="utf-8") as f:
    f.write(latex_code)

# ============================================================
# 11. VERSION COMPACTA SOLO CON LOS COEFICIENTES
# ============================================================
compact_table = corr_table[["System", "Pearson r", "Spearman rho"]].copy()
compact_table["Pearson r"] = compact_table["Pearson r"].map(lambda v: f"{v:.4f}")
compact_table["Spearman rho"] = compact_table["Spearman rho"].map(lambda v: f"{v:.4f}")

print("\n=== Compact table ===")
print(compact_table.to_string(index=False))